# Multi-Week Pipeline Validation

This notebook validates the telemetry analysis pipeline across multiple weeks to ensure:
1. **Append mode works correctly** - Historical records accumulate without duplicates
2. **Missing units handled properly** - Units with no data get InsufficientData status
3. **Time-series tracking** - machine_status and classified maintain history

**Test Configuration:**
- Client: `cda`
- Week Range: Week 40 2025 to Week 04 2026 (18 weeks)
- Baseline: Single baseline computed once (Week 40 2025, 112-day lookback)

In [1]:
# Import required modules
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

# Import telemetry modules
from src.telemetry import data_loader, data_cleaner, baseline, scoring, aggregation, output_writer
from src.utils.logger import logger

print("✓ All modules imported successfully")

✓ All modules imported successfully


## Configuration

Define the week range and baseline parameters.

In [2]:
# Pipeline configuration
CLIENT = 'cda'
BASELINE_LOOKBACK_DAYS = 7 * 16  # 112 days
BASELINE_VERSION = '20251201'  # Fixed baseline for all weeks

# Define week range: Week 40 2025 to Week 04 2026
weeks_to_process = []

# 2025: Weeks 40-53
for week in range(40, 54):
    weeks_to_process.append((week, 2025))

# 2026: Weeks 1-4
for week in range(1, 5):
    weeks_to_process.append((week, 2026))

print(f"Configuration:")
print(f"  Client: {CLIENT}")
print(f"  Total weeks to process: {len(weeks_to_process)}")
print(f"  Week range: Week {weeks_to_process[0][0]} {weeks_to_process[0][1]} to Week {weeks_to_process[-1][0]} {weeks_to_process[-1][1]}")
print(f"  Baseline: {BASELINE_VERSION} ({BASELINE_LOOKBACK_DAYS} days lookback)")
print(f"\nWeeks to process:")
for week, year in weeks_to_process:
    print(f"  - Week {week:02d}, {year}")

Configuration:
  Client: cda
  Total weeks to process: 18
  Week range: Week 40 2025 to Week 4 2026
  Baseline: 20251201 (112 days lookback)

Weeks to process:
  - Week 40, 2025
  - Week 41, 2025
  - Week 42, 2025
  - Week 43, 2025
  - Week 44, 2025
  - Week 45, 2025
  - Week 46, 2025
  - Week 47, 2025
  - Week 48, 2025
  - Week 49, 2025
  - Week 50, 2025
  - Week 51, 2025
  - Week 52, 2025
  - Week 53, 2025
  - Week 01, 2026
  - Week 02, 2026
  - Week 03, 2026
  - Week 04, 2026


---
## Step 1: Compute Baseline (Once)

Compute a single baseline that will be used for all weeks.

In [3]:
print("=" * 80)
print("BASELINE COMPUTATION")
print("=" * 80)

# Use first evaluation week for baseline computation
BASELINE_EVAL_WEEK, BASELINE_EVAL_YEAR = weeks_to_process[0]

# Load baseline training window
baseline_training_df = data_loader.load_baseline_training_window(
    client=CLIENT,
    evaluation_week=BASELINE_EVAL_WEEK,
    evaluation_year=BASELINE_EVAL_YEAR,
    lookback_days=BASELINE_LOOKBACK_DAYS
)

print(f"\n✓ Loaded {len(baseline_training_df):,} historical rows")
print(f"  Date range: {baseline_training_df['Fecha'].min()} to {baseline_training_df['Fecha'].max()}")
print(f"  Units: {baseline_training_df['Unit'].nunique()}")

# Get signal columns
signal_cols = data_loader.get_signal_columns(baseline_training_df)
print(f"\n✓ Identified {len(signal_cols)} signal columns")

# Clean baseline data
baseline_training_clean = data_cleaner.clean_telemetry_data(baseline_training_df, signal_cols)
print(f"✓ Baseline data cleaned: {len(baseline_training_clean):,} rows")

# Compute baseline percentiles
baseline_df = baseline.compute_baseline_percentiles(
    training_df=baseline_training_clean,
    signal_cols=signal_cols,
    baseline_date=BASELINE_VERSION
)
print(f"✓ Computed {len(baseline_df):,} baseline combinations")

# Save baseline
baseline_path = baseline.save_baseline(baseline_df, CLIENT)
print(f"✓ Baseline saved to: {baseline_path}")

# Load component mapping (needed for all weeks)
component_mapping = data_loader.load_component_mapping(CLIENT)
print(f"✓ Loaded mapping for {len(component_mapping)} components")

BASELINE COMPUTATION
2026-02-25 16:05:23,384 - telemetry - INFO - Loading baseline data from 2025-06-16 to 2025-10-05
2026-02-25 16:05:25,544 - telemetry - INFO - Loaded 1053790 rows, 11 units for baseline training

✓ Loaded 1,053,790 historical rows
  Date range: 2025-06-16 00:00:00 to 2025-10-05 00:00:00
  Units: 11

✓ Identified 18 signal columns
2026-02-25 16:05:25,571 - telemetry - INFO - Starting data cleaning pipeline
2026-02-25 16:05:25,858 - telemetry - INFO - Timestamp validation: 1053790 valid rows
2026-02-25 16:05:26,195 - telemetry - WARNING - Removed 2797 duplicate readings
2026-02-25 16:05:26,342 - telemetry - WARNING - Signals with >50% missing data: {'EngOilFltr': np.float64(91.55151366374467)}
2026-02-25 16:05:26,830 - telemetry - WARNING - Flagged 1196 extreme outliers as NaN
2026-02-25 16:05:26,832 - telemetry - INFO - Data cleaning complete: 1050993 rows (0.3% removed)
✓ Baseline data cleaned: 1,050,993 rows
2026-02-25 16:05:26,833 - telemetry - INFO - Computing ba

---
## Step 2: Process All Weeks

Loop through each week and execute the full pipeline.

In [4]:
print("\n" + "=" * 80)
print("MULTI-WEEK PROCESSING")
print("=" * 80)

processing_summary = []

for week_num, year_num in weeks_to_process:
    print(f"\n{'─' * 80}")
    print(f"Processing Week {week_num:02d}, {year_num}")
    print(f"{'─' * 80}")
    
    try:
        # 1. Load evaluation week
        current_df = data_loader.load_evaluation_week(
            client=CLIENT,
            week=week_num,
            year=year_num
        )
        print(f"  ✓ Loaded {len(current_df):,} rows")
        
        # 2. Clean data
        current_df_clean = data_cleaner.clean_telemetry_data(current_df, signal_cols)
        print(f"  ✓ Cleaned: {len(current_df_clean):,} rows (removed {len(current_df) - len(current_df_clean):,})")
        
        # 3. Score signals against baseline
        signal_evaluation_df = scoring.evaluate_signals(
            current_df=current_df_clean,
            baseline_df=baseline_df,
            signal_cols=signal_cols,
            component_mapping=component_mapping
        )
        print(f"  ✓ Evaluated {len(signal_evaluation_df):,} signal-unit combinations")
        
        # 4. Aggregate to components
        component_df = aggregation.aggregate_to_components(
            signal_evaluation_df=signal_evaluation_df,
            component_mapping=component_mapping,
            current_df=current_df_clean,
            evaluation_week=week_num,
            evaluation_year=year_num,
            baseline_version=BASELINE_VERSION
        )
        print(f"  ✓ Aggregated to {len(component_df):,} component-unit combinations")
        
        # 5. Get expected fleet for missing unit handling
        machine_status_path = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT / 'machine_status.parquet'
        expected_units = aggregation.get_expected_fleet(
            baseline_df=baseline_df,
            previous_machine_status_path=machine_status_path
        )
        
        # 6. Aggregate to machines
        machine_df = aggregation.aggregate_to_machines(
            component_df=component_df,
            evaluation_week=week_num,
            evaluation_year=year_num,
            baseline_version=BASELINE_VERSION,
            expected_units=expected_units,
            component_mapping=component_mapping
        )
        print(f"  ✓ Aggregated to {len(machine_df):,} machines")
        
        # 7. Write outputs
        output_writer.write_golden_outputs(
            machine_df=machine_df,
            component_df=component_df,
            client=CLIENT
        )
        print(f"  ✓ Golden layer outputs written")
        
        # Track summary
        processing_summary.append({
            'week': week_num,
            'year': year_num,
            'raw_rows': len(current_df),
            'clean_rows': len(current_df_clean),
            'signal_evaluations': len(signal_evaluation_df),
            'components': len(component_df),
            'machines': len(machine_df),
            'machines_normal': (machine_df['overall_status'] == 'Normal').sum(),
            'machines_alerta': (machine_df['overall_status'] == 'Alerta').sum(),
            'machines_anormal': (machine_df['overall_status'] == 'Anormal').sum(),
            'machines_insufficient': (machine_df['overall_status'] == 'InsufficientData').sum(),
            'components_normal': (component_df['component_status'] == 'Normal').sum(),
            'components_alerta': (component_df['component_status'] == 'Alerta').sum(),
            'components_anormal': (component_df['component_status'] == 'Anormal').sum(),
            'components_insufficient': (component_df['component_status'] == 'InsufficientData').sum(),
            'status': 'SUCCESS'
        })
        
    except Exception as e:
        print(f"  ✗ ERROR: {str(e)}")
        processing_summary.append({
            'week': week_num,
            'year': year_num,
            'status': f'ERROR: {str(e)}'
        })

print(f"\n{'=' * 80}")
print(f"✓ PROCESSING COMPLETE: {len(processing_summary)} weeks processed")
print(f"{'=' * 80}")


MULTI-WEEK PROCESSING

────────────────────────────────────────────────────────────────────────────────
Processing Week 40, 2025
────────────────────────────────────────────────────────────────────────────────
2026-02-25 16:05:32,393 - telemetry - INFO - Loading evaluation week: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\silver\cda\Telemetry_Wide_With_States\Week40Year2025.parquet
2026-02-25 16:05:32,412 - telemetry - INFO - Loaded 62402 rows, 10 units for Week 40 Year 2025
  ✓ Loaded 62,402 rows
2026-02-25 16:05:32,413 - telemetry - INFO - Starting data cleaning pipeline
2026-02-25 16:05:32,431 - telemetry - INFO - Timestamp validation: 62402 valid rows
2026-02-25 16:05:32,442 - telemetry - WARNING - Removed 1177 duplicate readings
2026-02-25 16:05:32,457 - telemetry - WARNING - Signals with >50% missing data: {'EngOilFltr': np.float64(100.0)}
2026-02-25 16:05:32,472 - telemetry - WARNING - Flagged 201 extreme outliers as NaN
2026-02-25 16:05:32,472 - telemetry

In [5]:
# Display processing summary
summary_df = pd.DataFrame(processing_summary)
summary_df['week_label'] = summary_df.apply(lambda x: f"W{x['week']:02d}-{x['year']}", axis=1)

print("\nProcessing Summary:")
print(summary_df[['week_label', 'machines', 'machines_normal', 'machines_alerta', 
                   'machines_anormal', 'machines_insufficient', 'status']].to_string(index=False))


Processing Summary:
week_label  machines  machines_normal  machines_alerta  machines_anormal  machines_insufficient                                                                                                                                                                status
  W40-2025      11.0              6.0              0.0               4.0                    1.0                                                                                                                                                               SUCCESS
  W41-2025      11.0              7.0              1.0               2.0                    1.0                                                                                                                                                               SUCCESS
  W42-2025      11.0              7.0              2.0               2.0                    0.0                                                                                                  

---
## Step 3: Validate Historical Tracking

Load the golden layer files and verify historical records accumulated correctly.

In [6]:
print("=" * 80)
print("HISTORICAL TRACKING VALIDATION")
print("=" * 80)

# Load machine_status
machine_status_path = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT / 'machine_status.parquet'
machine_status_df = pd.read_parquet(machine_status_path)

print(f"\nMachine Status (machine_status.parquet):")
print(f"  Total records: {len(machine_status_df):,}")
print(f"  Unique units: {machine_status_df['unit_id'].nunique()}")
print(f"  Unique weeks: {machine_status_df[['evaluation_week', 'evaluation_year']].drop_duplicates().shape[0]}")

# Check for duplicates
duplicates = machine_status_df.duplicated(subset=['unit_id', 'evaluation_week', 'evaluation_year'], keep=False)
if duplicates.any():
    print(f"  ⚠️ WARNING: Found {duplicates.sum()} duplicate records!")
    print(machine_status_df[duplicates][['unit_id', 'evaluation_week', 'evaluation_year', 'overall_status']].head(10))
else:
    print(f"  ✓ No duplicates found (deduplication working)")

# Show week coverage
week_coverage = machine_status_df[['evaluation_year', 'evaluation_week']].drop_duplicates().sort_values(['evaluation_year', 'evaluation_week'])
print(f"\n  Week coverage:")
for _, row in week_coverage.iterrows():
    count = len(machine_status_df[(machine_status_df['evaluation_week'] == row['evaluation_week']) & 
                                   (machine_status_df['evaluation_year'] == row['evaluation_year'])])
    print(f"    W{row['evaluation_week']:02d}-{row['evaluation_year']}: {count} units")

HISTORICAL TRACKING VALIDATION

Machine Status (machine_status.parquet):
  Total records: 187
  Unique units: 11
  Unique weeks: 17
  ✓ No duplicates found (deduplication working)

  Week coverage:
    W40-2025: 11 units
    W41-2025: 11 units
    W42-2025: 11 units
    W43-2025: 11 units
    W44-2025: 11 units
    W45-2025: 11 units
    W46-2025: 11 units
    W47-2025: 11 units
    W48-2025: 11 units
    W49-2025: 11 units
    W50-2025: 11 units
    W51-2025: 11 units
    W52-2025: 11 units
    W01-2026: 11 units
    W02-2026: 11 units
    W03-2026: 11 units
    W04-2026: 11 units


In [7]:
# Load classified
classified_path = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT / 'classified.parquet'
classified_df = pd.read_parquet(classified_path)

print(f"\nComponent Classification (classified.parquet):")
print(f"  Total records: {len(classified_df):,}")
print(f"  Unique units: {classified_df['unit_id'].nunique()}")
print(f"  Unique components: {classified_df['component'].nunique()}")
print(f"  Unique weeks: {classified_df[['evaluation_week', 'evaluation_year']].drop_duplicates().shape[0]}")

# Check for duplicates
duplicates = classified_df.duplicated(subset=['unit_id', 'component', 'evaluation_week', 'evaluation_year'], keep=False)
if duplicates.any():
    print(f"  ⚠️ WARNING: Found {duplicates.sum()} duplicate records!")
    print(classified_df[duplicates][['unit_id', 'component', 'evaluation_week', 'evaluation_year', 'component_status']].head(10))
else:
    print(f"  ✓ No duplicates found (deduplication working)")

# Show week coverage
week_coverage = classified_df[['evaluation_year', 'evaluation_week']].drop_duplicates().sort_values(['evaluation_year', 'evaluation_week'])
print(f"\n  Week coverage:")
for _, row in week_coverage.iterrows():
    count = len(classified_df[(classified_df['evaluation_week'] == row['evaluation_week']) & 
                               (classified_df['evaluation_year'] == row['evaluation_year'])])
    print(f"    W{row['evaluation_week']:02d}-{row['evaluation_year']}: {count} component-unit combinations")


Component Classification (classified.parquet):
  Total records: 724
  Unique units: 11
  Unique components: 4
  Unique weeks: 17
  ✓ No duplicates found (deduplication working)

  Week coverage:
    W40-2025: 40 component-unit combinations
    W41-2025: 40 component-unit combinations
    W42-2025: 44 component-unit combinations
    W43-2025: 44 component-unit combinations
    W44-2025: 44 component-unit combinations
    W45-2025: 44 component-unit combinations
    W46-2025: 44 component-unit combinations
    W47-2025: 44 component-unit combinations
    W48-2025: 44 component-unit combinations
    W49-2025: 44 component-unit combinations
    W50-2025: 44 component-unit combinations
    W51-2025: 44 component-unit combinations
    W52-2025: 40 component-unit combinations
    W01-2026: 40 component-unit combinations
    W02-2026: 44 component-unit combinations
    W03-2026: 40 component-unit combinations
    W04-2026: 40 component-unit combinations


In [9]:
# Check for InsufficientData cases
print("\nInsufficientData Cases:")

insufficient_machines = machine_status_df[machine_status_df['overall_status'] == 'InsufficientData']
print(f"  Machines with InsufficientData: {len(insufficient_machines)}")
if len(insufficient_machines) > 0:
    print(f"\n  Examples:")
    print(insufficient_machines[['unit_id', 'evaluation_week', 'evaluation_year', 
                                  'machine_score', 'total_components']].head(10).to_string(index=False))

insufficient_components = classified_df[classified_df['component_status'] == 'InsufficientData']
print(f"\n  Components with InsufficientData: {len(insufficient_components)}")
if len(insufficient_components) > 0:
    print(f"\n  Examples:")
    print(insufficient_components[['unit_id', 'component', 'evaluation_week', 'evaluation_year', 
                                    'component_score', 'triggering_signals']].head(10).to_string(index=False))


InsufficientData Cases:
  Machines with InsufficientData: 6

  Examples:
unit_id  evaluation_week  evaluation_year  machine_score  total_components
   T_11                3             2026            0.0                 4
   T_11                4             2026            0.0                 4
   T_13                1             2026            0.0                 4
   T_24               40             2025            0.0                 4
   T_24               41             2025            0.0                 4
   T_24               52             2025            0.0                 4

  Components with InsufficientData: 0


---
## Step 4: Time-Series Visualizations

Visualize machine and component status distributions over time.

In [10]:
# Prepare data for visualization
machine_status_df['week_label'] = machine_status_df.apply(
    lambda x: f"{x['evaluation_year']}-W{x['evaluation_week']:02d}", axis=1
)

classified_df['week_label'] = classified_df.apply(
    lambda x: f"{x['evaluation_year']}-W{x['evaluation_week']:02d}", axis=1
)

# Sort week labels properly
def sort_key(week_label):
    year, week = week_label.split('-W')
    return (int(year), int(week))

machine_status_df['sort_key'] = machine_status_df['week_label'].apply(sort_key)
classified_df['sort_key'] = classified_df['week_label'].apply(sort_key)

machine_status_df = machine_status_df.sort_values('sort_key')
classified_df = classified_df.sort_values('sort_key')

print("✓ Data prepared for visualization")

✓ Data prepared for visualization


In [11]:
# Visualization 1: Machine Status Distribution Over Time
print("Creating Machine Status Distribution chart...")

machine_status_counts = machine_status_df.groupby(['week_label', 'overall_status']).size().reset_index(name='count')

# Define color mapping
status_colors = {
    'Normal': '#28a745',      # Green
    'Alerta': '#ffc107',      # Yellow/Orange
    'Anormal': '#dc3545',     # Red
    'InsufficientData': '#6c757d'  # Gray
}

fig_machines = px.bar(
    machine_status_counts,
    x='week_label',
    y='count',
    color='overall_status',
    title='Machine Status Distribution Over Time',
    labels={
        'week_label': 'Week-Year',
        'count': 'Number of Units',
        'overall_status': 'Overall Status'
    },
    color_discrete_map=status_colors,
    category_orders={'overall_status': ['Normal', 'Alerta', 'Anormal', 'InsufficientData']},
    barmode='stack',
    height=500
)

fig_machines.update_layout(
    xaxis_tickangle=-45,
    font=dict(size=12),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig_machines.show()
print("✓ Machine status chart created")

Creating Machine Status Distribution chart...


✓ Machine status chart created


In [12]:
# Visualization 2: Component Status Distribution Over Time
print("Creating Component Status Distribution chart...")

component_status_counts = classified_df.groupby(['week_label', 'component_status']).size().reset_index(name='count')

fig_components = px.bar(
    component_status_counts,
    x='week_label',
    y='count',
    color='component_status',
    title='Component Status Distribution Over Time',
    labels={
        'week_label': 'Week-Year',
        'count': 'Number of Components',
        'component_status': 'Component Status'
    },
    color_discrete_map=status_colors,
    category_orders={'component_status': ['Normal', 'Alerta', 'Anormal', 'InsufficientData']},
    barmode='stack',
    height=500
)

fig_components.update_layout(
    xaxis_tickangle=-45,
    font=dict(size=12),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig_components.show()
print("✓ Component status chart created")

Creating Component Status Distribution chart...


✓ Component status chart created


---
## Step 5: Additional Analysis

Explore trends and patterns in the multi-week data.

In [13]:
# Calculate percentage distributions
print("=" * 80)
print("STATUS DISTRIBUTION ANALYSIS")
print("=" * 80)

# Machine status percentages by week
machine_pct = machine_status_df.groupby(['week_label', 'overall_status']).size().unstack(fill_value=0)
machine_pct_normalized = machine_pct.div(machine_pct.sum(axis=1), axis=0) * 100

print("\nMachine Status Distribution (%):")
print(machine_pct_normalized.round(1).to_string())

# Average status distribution across all weeks
print("\nAverage Machine Status Across All Weeks:")
avg_status = machine_status_df['overall_status'].value_counts(normalize=True) * 100
for status, pct in avg_status.items():
    print(f"  {status}: {pct:.1f}%")

STATUS DISTRIBUTION ANALYSIS

Machine Status Distribution (%):
overall_status  Alerta  Anormal  InsufficientData  Normal
week_label                                               
2025-W40           0.0     36.4               9.1    54.5
2025-W41           9.1     18.2               9.1    63.6
2025-W42          18.2     18.2               0.0    63.6
2025-W43          27.3     18.2               0.0    54.5
2025-W44           0.0     18.2               0.0    81.8
2025-W45           0.0     18.2               0.0    81.8
2025-W46           9.1     18.2               0.0    72.7
2025-W47           9.1     18.2               0.0    72.7
2025-W48           9.1     18.2               0.0    72.7
2025-W49          18.2     18.2               0.0    63.6
2025-W50           9.1     18.2               0.0    72.7
2025-W51          27.3     18.2               0.0    54.5
2025-W52          27.3     27.3               9.1    36.4
2026-W01          45.5     27.3               9.1    18.2
2026-W02 

In [14]:
# Component status percentages by week
component_pct = classified_df.groupby(['week_label', 'component_status']).size().unstack(fill_value=0)
component_pct_normalized = component_pct.div(component_pct.sum(axis=1), axis=0) * 100

print("\nComponent Status Distribution (%):")
print(component_pct_normalized.round(1).to_string())

# Average status distribution across all weeks
print("\nAverage Component Status Across All Weeks:")
avg_comp_status = classified_df['component_status'].value_counts(normalize=True) * 100
for status, pct in avg_comp_status.items():
    print(f"  {status}: {pct:.1f}%")


Component Status Distribution (%):
component_status  Alerta  Anormal  Normal
week_label                               
2025-W40            12.5     27.5    60.0
2025-W41            12.5     17.5    70.0
2025-W42            25.0     22.7    52.3
2025-W43            22.7     20.5    56.8
2025-W44            15.9     18.2    65.9
2025-W45            18.2     11.4    70.5
2025-W46            15.9     20.5    63.6
2025-W47            25.0     20.5    54.5
2025-W48            15.9     20.5    63.6
2025-W49            27.3     20.5    52.3
2025-W50            18.2     18.2    63.6
2025-W51            29.5     20.5    50.0
2025-W52            32.5     32.5    35.0
2026-W01            45.0     27.5    27.5
2026-W02            29.5     22.7    47.7
2026-W03            45.0     27.5    27.5
2026-W04            42.5     27.5    30.0

Average Component Status Across All Weeks:
  Normal: 52.8%
  Alerta: 25.3%
  Anormal: 22.0%


In [15]:
# Identify units with status changes
print("\n" + "=" * 80)
print("UNIT STATUS EVOLUTION")
print("=" * 80)

# Find units that changed status during the period
unit_status_evolution = machine_status_df.pivot_table(
    index='unit_id',
    columns='week_label',
    values='overall_status',
    aggfunc='first'
)

# Count unique statuses per unit
status_changes = unit_status_evolution.apply(lambda x: x.nunique(), axis=1)
units_with_changes = status_changes[status_changes > 1].sort_values(ascending=False)

print(f"\nUnits with status changes: {len(units_with_changes)}")
print(f"Units with stable status: {len(status_changes[status_changes == 1])}")

if len(units_with_changes) > 0:
    print(f"\nTop 10 units with most status changes:")
    for unit_id, change_count in units_with_changes.head(10).items():
        statuses = unit_status_evolution.loc[unit_id].dropna().unique()
        print(f"  {unit_id}: {change_count} different statuses - {list(statuses)}")


UNIT STATUS EVOLUTION

Units with status changes: 8
Units with stable status: 3

Top 10 units with most status changes:
  T_13: 4 different statuses - ['Normal', 'Anormal', 'InsufficientData', 'Alerta']
  T_11: 3 different statuses - ['Normal', 'Alerta', 'InsufficientData']
  T_12: 3 different statuses - ['Normal', 'Alerta', 'Anormal']
  T_15: 3 different statuses - ['Anormal', 'Normal', 'Alerta']
  T_24: 3 different statuses - ['InsufficientData', 'Normal', 'Alerta']
  T_16: 3 different statuses - ['Anormal', 'Normal', 'Alerta']
  T_14: 2 different statuses - ['Normal', 'Alerta']
  T_17: 2 different statuses - ['Normal', 'Alerta']


In [16]:
# Machine score trends
print("\n" + "=" * 80)
print("SEVERITY SCORE TRENDS")
print("=" * 80)

# Average machine score by week
avg_machine_scores = machine_status_df.groupby('week_label')['machine_score'].agg(['mean', 'median', 'max'])
print("\nMachine Score Statistics by Week:")
print(avg_machine_scores.round(2).to_string())

# Visualize score trends
fig_scores = go.Figure()

fig_scores.add_trace(go.Scatter(
    x=avg_machine_scores.index,
    y=avg_machine_scores['mean'],
    mode='lines+markers',
    name='Mean Score',
    line=dict(color='#007bff', width=2)
))

fig_scores.add_trace(go.Scatter(
    x=avg_machine_scores.index,
    y=avg_machine_scores['median'],
    mode='lines+markers',
    name='Median Score',
    line=dict(color='#28a745', width=2)
))

fig_scores.add_trace(go.Scatter(
    x=avg_machine_scores.index,
    y=avg_machine_scores['max'],
    mode='lines+markers',
    name='Max Score',
    line=dict(color='#dc3545', width=2, dash='dash')
))

fig_scores.update_layout(
    title='Machine Severity Score Trends',
    xaxis_title='Week-Year',
    yaxis_title='Machine Score',
    xaxis_tickangle=-45,
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig_scores.show()
print("✓ Score trend chart created")


SEVERITY SCORE TRENDS

Machine Score Statistics by Week:
            mean  median  max
week_label                   
2025-W40    0.85    0.30  2.8
2025-W41    0.56    0.12  2.8
2025-W42    0.85    0.42  2.8
2025-W43    0.79    0.40  2.8
2025-W44    0.63    0.12  2.8
2025-W45    0.50    0.12  2.8
2025-W46    0.73    0.30  2.8
2025-W47    0.80    0.42  2.8
2025-W48    0.74    0.30  2.8
2025-W49    0.81    0.42  2.8
2025-W50    0.72    0.30  2.8
2025-W51    0.88    0.42  2.8
2025-W52    1.11    0.84  2.8
2026-W01    1.05    0.72  2.8
2026-W02    0.93    0.72  2.8
2026-W03    1.07    0.72  2.8
2026-W04    1.04    0.72  2.8


✓ Score trend chart created


In [17]:
# Top problematic units
print("\n" + "=" * 80)
print("TOP PROBLEMATIC UNITS")
print("=" * 80)

# Calculate average score per unit across all weeks
top_units = machine_status_df.groupby('unit_id').agg({
    'machine_score': 'mean',
    'overall_status': lambda x: (x.isin(['Anormal', 'Alerta'])).sum(),
    'evaluation_week': 'count'
}).rename(columns={
    'machine_score': 'avg_score',
    'overall_status': 'weeks_with_issues',
    'evaluation_week': 'total_weeks'
})

top_units['issue_rate'] = (top_units['weeks_with_issues'] / top_units['total_weeks']) * 100
top_units = top_units.sort_values('avg_score', ascending=False)

print(f"\nTop 15 Units by Average Machine Score:")
print(top_units.head(15).round(2).to_string())


TOP PROBLEMATIC UNITS

Top 15 Units by Average Machine Score:
         avg_score  weeks_with_issues  total_weeks  issue_rate
unit_id                                                       
T_9           2.80                 17           17      100.00
T_18          2.51                 17           17      100.00
T_12          1.13                 14           17       82.35
T_14          0.47                  6           17       35.29
T_11          0.42                  5           17       29.41
T_13          0.41                  4           17       23.53
T_24          0.40                  5           17       29.41
T_16          0.34                  2           17       11.76
T_15          0.28                  4           17       23.53
T_17          0.22                  3           17       17.65
T_10          0.11                  0           17        0.00


---
## Validation Summary

**Key Findings:**
1. ✅ Historical tracking working - both files accumulate records
2. ✅ Deduplication working - no duplicate (unit, week, year) combinations
3. ✅ Missing unit handling - InsufficientData status appears when expected
4. ✅ Time-series queries functional - can track evolution over weeks

**Next Steps:**
- Use this validated data for dashboard development
- Implement alerting based on status changes
- Apply forecasting models to predict future status
- Fine-tune thresholds based on observed distributions